These are my notes post the let's build gpt video.
## Self-Attention block:
We basically have in an attention block: 
- an attention sub-block: x1 = softmax(Q K^T / sqrt (d_k) + M ) V 
- adding the residual (x = x + x1) 
- an MLP feedforward sub-block : x2 
- adding the residual x = x + x2. 

in karpathy's vid, he did layer norm before the attention sub-block and added a dropout at after the second risidual.

In [2]:
import torch
import torch.nn as nn
from torch.nn import functional as F



In the following, we suppose we have 4 heads in our attention block, each token is represented by a 32-dimensional embeding vector, our max token window is 12 and we have 8 batches.

In [ ]:
# setting:
B,H,T,n_embed = 8,4,12,32
d_k = n_embed // H
#x here is the input to our self-attention subblock.
x = torch.randn([B,T,n_embed]) #B,H,T,n_embed


#we now layer norm (unbiased = False to match the nn.Layernorm):
x = (x - x.mean(dim=-1, keepdim=True))/(torch.sqrt(x.var(dim=-1, keepdim=True, unbiased=False)+ 1e-5) )
print(x.mean(), x.var())

print(x.shape)

Q = nn.Linear(n_embed, H*d_k, bias=False)
K = nn.Linear(n_embed, H*d_k, bias=False)
V = nn.Linear(n_embed, H*d_k, bias=False)

query =Q(x)
query = query.view(B,T,H,d_k).transpose(1,2)
key =K(x)
key = key.view(B,T,H,d_k).transpose(1,2)
value = V(x)
value = value.view(B,T,H,d_k).transpose(1,2)


print (query.shape, key.shape, value.shape)

# we now calculate the product:
prod1 = query @ key.transpose(-2,-1) / (d_k)**(1/2)
# we create the masc:
tril = torch.tril(torch.ones(T,T))
masc = tril.masked_fill(tril == 0 , -float("inf"))
masc = masc.masked_fill(masc == 1 , 0)
# we sum and apply the soft max:
print(prod1.shape, masc.shape, value.shape)
x1 = F.softmax(prod1 + masc, dim = -1) @ value
print(x1.shape)




#Now we lay the W_O subblock and the risidual: 
W_O = nn.Linear(n_embed, n_embed )
x = x + W_O(x1.transpose(1,2).reshape(B,T,n_embed))

#one more layernorm:
ln2 = nn.LayerNorm(n_embed)
x = ln2(x)
# now for the feed forward:
FF = nn.Sequential(nn.Linear(n_embed, 4*n_embed), nn.GELU(), nn.Linear(4 * n_embed, n_embed))
x = FF(x) + x



tensor(-5.5879e-09) tensor(1.0003)
torch.Size([8, 12, 32])
torch.Size([8, 4, 12, 8]) torch.Size([8, 4, 12, 8]) torch.Size([8, 4, 12, 8])
torch.Size([8, 4, 12, 12]) torch.Size([12, 12]) torch.Size([8, 4, 12, 8])
torch.Size([8, 4, 12, 8])


when doing Q = nn.Linear(n_embed, d_k, bias=False) we're not really dividing the vector of imbeding into 4 parts exactly but more like, creating linear transformations to four $R^{\frac{n_embed}{H}}$ vectors.

Now we create the self attention head:

In [ ]:
# setting:
B,H,T,n_embed = 8,4,12,32



#x here is the input to our self-attention subblock.
x = torch.randn([B,T,n_embed]) #B,T,n_embed

class SelfAttention(nn.Module): #we inherit from nnmodule to integrate into the parameters and other useful stuff.

    def __init__(self, n_embed, Heads):
        super().__init__() # necessary to create QKV etc
        self.Heads = Heads
        self.n_embed = n_embed
        self.Q = nn.Linear(n_embed,n_embed, bias=False)
        self.K = nn.Linear(n_embed,n_embed, bias=False)
        self.V = nn.Linear(n_embed,n_embed, bias=False)
        self.W_O = nn.Linear(n_embed, n_embed )
        self.ln1 = nn.LayerNorm(n_embed)
        self.ln2 = nn.LayerNorm(n_embed)
        self.FF = nn.Sequential(nn.Linear(n_embed, 4*n_embed), nn.GELU(), nn.Linear(4 * n_embed, n_embed))
        

    
        
        

    def st(self,x):
        B,T,n_embed = x.shape
        H = self.Heads
        d_k = n_embed // H
        #we now layer norm (unbiased = False to match the nn.Layernorm):
        #x = (x - x.mean(dim=-1, keepdim=True))/(torch.sqrt(x.var(dim=-1, keepdim=True, unbiased=False)+ 1e-5) )
        
        x = self.ln1(x)
        print(x.shape)

        

        query =self.Q(x)
        query = query.view(B,T,H,d_k).transpose(1,2)
        key =self.K(x)
        key = key.view(B,T,H,d_k).transpose(1,2)
        value =self.V(x)
        value = value.view(B,T,H,d_k).transpose(1,2)


        print (query.shape, key.shape, value.shape)

        # we now calculate the product:
        prod1 = query @ key.transpose(-2,-1) / (d_k)**(1/2)
        # we create the masc:
        tril = torch.tril(torch.ones(T,T))
        masc = tril.masked_fill(tril == 0 , -float("inf"))
        masc = masc.masked_fill(masc == 1 , 0)
        # we sum and apply the soft max:
        print(prod1.shape, masc.shape, value.shape)
        x1 = F.softmax(prod1 + masc, dim = -1) @ value
        print(x1.shape)




        #Now we lay the W_O subblock and the risidual: 
        
        x = x + self.W_O(x1.transpose(1,2).reshape(B,T,n_embed))
        return x

    def ff(self,x):
        B,T,n_embed = x.shape

        #one more layernorm:
        
        x = self.ln2(x)
        # now for the feed forward:
        
        x = self.FF(x) + x
        return x

    def forward(self,x): # the nn.module will handle calling
        x = self.st(x)
        x = self.ff(x)
        return x






Let's try:


In [29]:
x = torch.randn([3,4,20]) #B,H,T,n_embed
SA = SelfAttention(20,5)
x = SA(x)

print(x.shape)

torch.Size([3, 4, 20])
torch.Size([3, 5, 4, 4]) torch.Size([3, 5, 4, 4]) torch.Size([3, 5, 4, 4])
torch.Size([3, 5, 4, 4]) torch.Size([4, 4]) torch.Size([3, 5, 4, 4])
torch.Size([3, 5, 4, 4])
torch.Size([3, 4, 20])


In [30]:
x = torch.randn([12,128,128]) #B,H,T,n_embed
SA = SelfAttention(128,32)
x = SA(x)

print(x.shape)

torch.Size([12, 128, 128])
torch.Size([12, 32, 128, 4]) torch.Size([12, 32, 128, 4]) torch.Size([12, 32, 128, 4])
torch.Size([12, 32, 128, 128]) torch.Size([128, 128]) torch.Size([12, 32, 128, 4])
torch.Size([12, 32, 128, 4])
torch.Size([12, 128, 128])


Some remarks / important errors to avoid :
- All parameters should be initialized in your `__init__`, and your class should inherit from `nn.Modules`
- We don't implement a `__call__` but instead just a `forward`, we let the `nn.Modules` handle calling.
- When doing layer normalization, remember that you're doing it on the last dim (`n_embed`).
- Make sure your dim handling is correct (especially the transposes).
- One can also add a dropout to avoid overfit.